In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import json
import spacy
import os

In [ ]:
BASE_FOLDER = "/content/drive/MyDrive/Frames_experiment"
ANNOTATED_FILE = os.path.join(BASE_FOLDER, "dataset_annotated.jsonl")
FILTERED_FILE = os.path.join(BASE_FOLDER, "filtered_dataset.jsonl") #option file - if all sentences are annotated, then this should not be needed
BIO_FILE = os.path.join(BASE_FOLDER, "dataset_bio.conll")

In [ ]:
#Optional script for filtering annotated files

with open(ANNOTATED_FILE, "r", encoding="utf-8") as f, \
     open(FILTERED_FILE, "w", encoding="utf-8") as out:

    for line in f:
        item = json.loads(line)
        sent_id = item["id"]
        labels = item["label"]

        if labels or (1401 <= sent_id <= 1599):
            out.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Filtered dataset created:", FILTERED_FILE)

In [ ]:
!pip install -q spacy
!python -m spacy download pt_core_news_sm

nlp = spacy.load("pt_core_news_sm")

In [ ]:
#Convert to BIO Format

with open(FILTERED_FILE, "r", encoding="utf-8") as f, \
     open(BIO_FILE, "w", encoding="utf-8") as out:

    for line in f:
        item = json.loads(line)
        text = item["text"]
        labels = item["label"]

        doc = nlp(text)
        tokens = [token.text for token in doc]
        starts = [token.idx for token in doc]
        ends = [token.idx + len(token.text) for token in doc]

        tags = ["O"] * len(tokens)

        for start, end, label in labels:
            first = True
            for i, (t_start, t_end) in enumerate(zip(starts, ends)):
                if t_start >= start and t_end <= end:
                    if first:
                        tags[i] = "B-" + label
                        first = False
                    else:
                        tags[i] = "I-" + label

        for token, tag in zip(tokens, tags):
            out.write(f"{token}\t{tag}\n")
        out.write("\n")

print("BIO dataset created:", BIO_FILE)